# 音のプログラミング 第2回: エンベロープとADSR

**前回の復習:** サイン波の生成、周波数と音の高さ、サンプリングの基礎

**学習目標:**
- エンベロープの概念を理解する
- ADSR（Attack, Decay, Sustain, Release）を体験する
- エンベロープで自然な楽器音を模倣する
- クリック音の原因と対処法を知る

**所要時間:** 90分

## 🛠️ 環境セットアップ

In [ ]:
# 🛠️ 環境セットアップ
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    !pip install japanize-matplotlib
    !git clone https://github.com/ggszk/simple-audio-programming.git
    sys.path.append('/content/simple-audio-programming')
    import japanize_matplotlib
else:
    print("🔧 ローカル環境を設定中...")
    sys.path.append('..')
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("✅ セットアップ完了")

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import sine_wave, adsr, linear_envelope, save_audio, AudioSignal
from audio_lib.synthesis.note_utils import note_to_frequency, note_name_to_number
from audio_lib.notebook import play_sound, plot_waveform

## 🎵 クリック音を体験しよう

サイン波をそのまま再生すると、音の始まりや終わりで「プツッ」というクリック音が発生することがあります。

これは波形の**始点は sin(0)=0 で問題ありませんが、終点が0でない場合に不連続点**ができるためです。

In [ ]:
# クリック音を体験
raw_signal = sine_wave(440, 1.0)

print(f"波形の開始値: {raw_signal.data[0]:.6f}")
print(f"波形の終了値: {raw_signal.data[-1]:.6f}")

display(play_sound(raw_signal, "エンベロープなし（クリック音あり）"))

## 🎵 エンベロープとは？

実際の楽器の音は、時間とともに音量が変化します：

- **ピアノ**: 弾いた瞬間が最大 → ゆっくり減衰
- **バイオリン**: ゆっくり立ち上がり → 一定に持続
- **フルート**: 息を吹き込んで徐々に大きく → 息を止めて減衰

この「音量の時間変化」を**エンベロープ**（包絡線）と呼びます。

## 🎯 実習1: シンプルなフェードイン・フェードアウト

In [ ]:
# リニアエンベロープ（直線的なフェードイン・アウト）
envelope = linear_envelope(1.0, fade_in=0.1, fade_out=0.1, sample_rate=44100)

# エンベロープの形を表示
import matplotlib.pyplot as plt

time = np.linspace(0, 1.0, len(envelope))
plt.figure(figsize=(12, 3))
plt.plot(time, envelope.data, 'r-', linewidth=2)
plt.title("リニアエンベロープ（フェードイン 0.1秒 / フェードアウト 0.1秒）", fontsize=14)
plt.xlabel("時間 (秒)")
plt.ylabel("振幅")
plt.grid(True, alpha=0.3)
plt.show()

音の始まりと終わりがなめらかに0に向かっています。これでクリック音を防げます。

In [ ]:
# エンベロープを適用して再生
raw_signal = sine_wave(440, 1.0)
smooth_signal = raw_signal * envelope

display(play_sound(smooth_signal, "エンベロープあり（クリック音なし）"))

クリック音が消えて、なめらかな音になりました。

### エンベロープ適用前後の波形を比較してみよう

In [ ]:
# エンベロープ適用前後の波形比較
raw_signal = sine_wave(440, 1.0)
envelope = linear_envelope(1.0, fade_in=0.1, fade_out=0.1, sample_rate=44100)
smooth_signal = raw_signal * envelope

plot_waveform(raw_signal, duration=0.1, title="エンベロープなし（冒頭 0.1秒）")
plot_waveform(smooth_signal, duration=0.1, title="エンベロープあり（冒頭 0.1秒）")

## 🎵 ADSR エンベロープ

より本格的なエンベロープが**ADSR**です。4つのフェーズから構成されます：

```
   振幅
    ^
  1 |    /\
    |   /  \___________
  S |  /    D          \
    | / A               \ R
  0 +---+--+----------+--+---> 時間
      A   D   Sustain    R
```

- **Attack（アタック）**: 音が立ち上がるまでの時間
- **Decay（ディケイ）**: 最大音量からサスティンレベルまで下がる時間
- **Sustain（サスティン）**: 鍵盤を押している間の音量レベル（0〜1）
- **Release（リリース）**: 鍵盤を離してから音が消えるまでの時間

## 🎯 実習2: ADSRエンベロープを体験しよう

In [ ]:
# ADSR エンベロープを作成
duration = 2.0
envelope = adsr(duration, attack=0.1, decay=0.2, sustain=0.7, release=0.5)

# エンベロープの形を表示
time = np.linspace(0, duration, len(envelope))
plt.figure(figsize=(12, 4))
plt.plot(time, envelope.data, 'r-', linewidth=2)
plt.title("ADSR エンベロープ", fontsize=14)
plt.xlabel("時間 (秒)")
plt.ylabel("振幅")

# フェーズの区切りを表示
plt.axvline(x=0.1, color='gray', linestyle='--', alpha=0.5, label='Attack終了')
plt.axvline(x=0.3, color='gray', linestyle='--', alpha=0.5, label='Decay終了')
plt.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5, label='Release開始')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ADSRエンベロープを適用して再生
signal = sine_wave(440, 2.0)
shaped_signal = signal * envelope

display(play_sound(shaped_signal, "ADSR適用 (A=0.1, D=0.2, S=0.7, R=0.5)"))

シンセサイザーのような音になりました。

## 🎯 実習3: ADSRパラメータを変えて楽器を模倣しよう

In [ ]:
# ピアノ風: 速いアタック、長いリリース、低いサスティン
signal = sine_wave(440, 3.0)
piano_env = adsr(3.0, attack=0.01, decay=0.5, sustain=0.3, release=1.5)
piano_sound = signal * piano_env

display(play_sound(piano_sound, "ピアノ風 (A=0.01, D=0.5, S=0.3, R=1.5)"))

In [ ]:
# オルガン風: 短いアタック、ほぼ減衰なし、フルサスティン
signal = sine_wave(440, 3.0)
organ_env = adsr(3.0, attack=0.05, decay=0.1, sustain=1.0, release=0.1)
organ_sound = signal * organ_env

display(play_sound(organ_sound, "オルガン風 (A=0.05, D=0.1, S=1.0, R=0.1)"))

In [ ]:
# ストリングス風: ゆっくりアタック、高いサスティン
signal = sine_wave(440, 3.0)
string_env = adsr(3.0, attack=0.3, decay=0.3, sustain=0.8, release=0.5)
string_sound = signal * string_env

display(play_sound(string_sound, "ストリングス風 (A=0.3, D=0.3, S=0.8, R=0.5)"))

同じサイン波でも、エンベロープを変えるだけでまったく違う楽器に聞こえます。

## 🎯 実習4: エンベロープ付きメロディーを作ろう

In [ ]:
# 「きらきら星」の最初の部分をADSRエンベロープ付きで演奏
melody = [
    ("C4", 0.5), ("C4", 0.5), ("G4", 0.5), ("G4", 0.5),
    ("A4", 0.5), ("A4", 0.5), ("G4", 1.0),
    ("F4", 0.5), ("F4", 0.5), ("E4", 0.5), ("E4", 0.5),
    ("D4", 0.5), ("D4", 0.5), ("C4", 1.0),
]

melody_data = []
for note_name, dur in melody:
    freq = note_to_frequency(note_name_to_number(note_name))
    sig = sine_wave(freq, dur)
    env = adsr(dur, attack=0.01, decay=0.1, sustain=0.7, release=0.15)
    note_sound = sig * env
    melody_data.append(note_sound.data)

melody_signal = AudioSignal(np.concatenate(melody_data), 44100)
display(play_sound(melody_signal, "きらきら星（エンベロープ付き）"))

自然な発音と消音で、きれいなメロディーになりました。

## 🎯 実習5: エンベロープありなしの比較

In [ ]:
# エンベロープなし版（比較用）
melody_raw = []
for note_name, dur in melody:
    freq = note_to_frequency(note_name_to_number(note_name))
    sig = sine_wave(freq, dur)
    melody_raw.append(sig.data)

raw_melody = AudioSignal(np.concatenate(melody_raw), 44100)
display(play_sound(raw_melody, "きらきら星（エンベロープなし）"))
display(play_sound(melody_signal, "きらきら星（エンベロープ付き）"))

エンベロープなしの版では音と音の境界でクリック音が発生し、機械的に聞こえます。
エンベロープ付きの版は音の立ち上がりと減衰が自然で、心地よく聞こえます。

## 🏆 チャレンジ課題

In [ ]:
# チャレンジ1: 自分でADSRパラメータを調整してみよう
my_attack = 0.05    # ← 変更してみよう (0.001 〜 1.0)
my_decay = 0.3      # ← 変更してみよう (0.1 〜 2.0)
my_sustain = 0.5    # ← 変更してみよう (0.0 〜 1.0)
my_release = 0.8    # ← 変更してみよう (0.1 〜 3.0)

signal = sine_wave(440, 3.0)
my_env = adsr(3.0, attack=my_attack, decay=my_decay,
              sustain=my_sustain, release=my_release)
my_sound = signal * my_env

# エンベロープの形を表示
time = np.linspace(0, 3.0, len(my_env))
plt.figure(figsize=(12, 3))
plt.plot(time, my_env.data, 'r-', linewidth=2)
plt.title(f"カスタムADSR (A={my_attack}, D={my_decay}, S={my_sustain}, R={my_release})")
plt.xlabel("時間 (秒)")
plt.ylabel("振幅")
plt.grid(True, alpha=0.3)
plt.show()

display(play_sound(my_sound, "カスタムADSR"))

In [ ]:
# チャレンジ2: 自分だけのメロディーを作ろう
my_melody = [
    ("C4", 0.5), ("E4", 0.5), ("G4", 0.5), ("C5", 1.0),
]

my_melody_data = []
for note_name, dur in my_melody:
    freq = note_to_frequency(note_name_to_number(note_name))
    sig = sine_wave(freq, dur)
    env = adsr(dur, attack=0.01, decay=0.1, sustain=0.7, release=0.15)
    my_melody_data.append((sig * env).data)

my_melody_signal = AudioSignal(np.concatenate(my_melody_data), 44100)
display(play_sound(my_melody_signal, "カスタムメロディー"))

## 📚 今日のまとめ

### 学んだこと
1. **エンベロープ**は音の時間的な音量変化を制御する
2. **ADSR**はAttack, Decay, Sustain, Releaseの4パラメータで構成
3. ADSRパラメータを変えるだけで異なる楽器の特徴を模倣できる
4. エンベロープなしの音は**クリック音**が発生する（波形の不連続が原因）
5. エンベロープを使うことで自然で心地よい音になる

### 使ったライブラリ関数
- `adsr(duration, attack, decay, sustain, release)` — ADSRエンベロープ
- `linear_envelope(duration, fade_in, fade_out)` — リニアエンベロープ
- `signal * envelope` — エンベロープの適用

### 次回予告
第3回では**周波数分析（FFTとスペクトログラム）**を学びます。
波形の違い（サイン波 / 矩形波 / のこぎり波）がなぜ異なる音色に聞こえるのか、その仕組みを理解します。